# Bayesian Unfolding

This notebook contains the traditional **Bayesian unfolding** of the smeared data.

At first, we need to import ROOT with the RooUnfold library, and select data samples for training (creating the response matrix and unfolding matrix) and unfolding.

In [1]:
import ROOT
ROOT.gSystem.Load("libRooUnfold") # load RooUnfold
ROOT.gStyle.SetOptStat(0) # disables the stat boxes in plots

# select data for training and unfolding by their seed
seedTraining = 2
seedUnfolding = 22

/home/javkovsky/anaconda3/envs/star-bachelor-env/lib/python3.11/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /home/javkovsky/anaconda3/envs/star-bachelor-env/include/site/python3.11); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "


### 1D Distributions

We load the histograms generated in the plot-ea.ipynb notebook.

In [2]:
# open files for reading
fileTraining = ROOT.TFile(f"data/{seedTraining}/events{seedTraining}_plots.root", "READ")
fileUnfolding = ROOT.TFile(f"data/{seedUnfolding}/events{seedUnfolding}_plots.root", "READ") # a different simulation -- represents the measured data

distributions = {'Nch': 'Multiplicity', 'S0pT1': 'Unweighted Spherocity'} # dictionary of distributions we want to unfold (e.g., multiplicity, spherocity, pT spectra, ...), syntax: abbreviation: full name
histos = {} # create an empty dictionary where we will store our histograms

# fill the distributions dictionary
for abbrev, name in distributions.items():
    # syntax: abbreviation: [name, true training data, smeared training data, response matrix from training data, independent true data, smeared data to be unfolded]
    histos[abbrev] = [fileTraining.Get(f"histo{seedTraining}{abbrev}{kind}") for kind in ["True", "Smeared", "RM"]] # fill in the training data
    histos[abbrev] += [fileUnfolding.Get(f"histo{seedUnfolding}{abbrev}{kind}") for kind in ["True", "Smeared"]] # append the data to be unfolded
    histos[abbrev].insert(0, name) # prepend the full name of the observable

Afterwards, we construct response matrices compatible with RooUnfold for each observable from the training data. Then, we proceed with the Bayesian unfolding of the measured data and plot all distributions for comparison. Lastly, we perform a closure test to evaluate the quality of unfolding.

In [3]:
# create a dictionary where unfolding matrices (UMs) will be stored and a dictionary to store unfolded distributions
UMs = {}
histosUnfolded = {}

# loop over different observables
for abbrev, data in histos.items():
    # define histogram variables from the histos dictionary
    name = data[0]
    histoTrainingTrue = data[1]
    histoTrainingSmeared = data[2]
    histoTrainingRM = data[3]
    histoUnfoldingTrue = data[4]
    histoUnfoldingSmeared = data[5]

    # ---------
    # UNFOLDING
    # ---------

    # create a RooUnfold response matrix
    response = ROOT.RooUnfoldResponse(histoTrainingSmeared, histoTrainingTrue, histoTrainingRM)

    # Bayesian unfolding with nIter iterations
    nIter = 5 # define the number of iterations
    unfolded = ROOT.RooUnfoldBayes(response, histoTrainingSmeared) # initialize the unfolding object
    
    unfolded.SetIterations(nIter - 1) # perform nIter-1 iterations
    histoPenultimate = unfolded.Hunfold().Clone("histoPenultimate") # extract the histogram after the penultimate iteration (we must use the .Clone() method otherwise defining histoUnfolded below would overwrite this histogram)
    histoPenultimate.SetDirectory(0) # makes sure that Python takes memory management ownership of the object

    unfolded.SetIterations(nIter) # perform nIter iterations
    histoUnfolded = unfolded.Hunfold().Clone("histoUnfolded") # extract the unfolded histogram (we must once again use the .Clone() method for several reasons: .Hunfold() returns a pointer to an internal histogram managed by the RooUnfold object; if we modified (e.g., scaled) the histogram without cloning it, it would alter the internal state of the unfolding engine and if we then wanted to extract some more information about the RooUnfold object (e.g., a covariance matrix), the math would be broken)
    histoUnfolded.SetDirectory(0)

    UMs[abbrev] = unfolded.UnfoldingMatrix() # extract the unfolding matrix (UM)
    histosUnfolded[abbrev] = histoUnfolded # extract the unfolded histogram

    # --------------
    # UNFOLDING PLOT
    # --------------

    # set up canvas for drawing
    canvasUnfolding = ROOT.TCanvas(f"canvasUnfolding{abbrev}", f"Bayesian Unfolding of {name}", 800, 600)
    canvasUnfolding.SetLogy() 

    # normalize the histograms
    for histo in [histoUnfoldingTrue, histoUnfoldingSmeared, histoPenultimate, histoUnfolded]:
        integral = histo.Integral()
        if integral > 0:
            histo.Scale(1.0 / integral)

    # customize the histograms (to customize title, labels, ..., we have to modify the first drawn histogram)
    histoUnfoldingTrue.SetTitle(f"Bayesian Unfolding of {name};Values;Normalized Events") # set title and labels of axes

    maxVal = max(histoUnfoldingTrue.GetMaximum(), histoUnfoldingSmeared.GetMaximum(), histoPenultimate.GetMaximum(), histoUnfolded.GetMaximum())
    histoUnfoldingTrue.SetMaximum(maxVal * 4.5) # scale the y-axis to prevent cut-off

    histoUnfoldingTrue.SetLineColor(ROOT.kGreen)
    histoUnfoldingSmeared.SetLineColor(ROOT.kBlue)
    histoPenultimate.SetLineColor(ROOT.kOrange)
    histoUnfolded.SetLineColor(ROOT.kRed)

    histoPenultimate.SetMarkerStyle(ROOT.kOpenSquare)
    histoPenultimate.SetMarkerColor(ROOT.kOrange)
    histoPenultimate.SetMarkerSize(0.8)

    histoUnfolded.SetMarkerStyle(ROOT.kFullCircle)
    histoUnfolded.SetMarkerColor(ROOT.kRed)
    histoUnfolded.SetMarkerSize(0.5)

    # drawing
    histoUnfoldingTrue.Draw("HIST")
    histoUnfoldingSmeared.Draw("HIST SAME") # "SAME" tells ROOT to draw the histogram on top of the previous one
    histoPenultimate.Draw("PE SAME") # "P" forces a point scatter
    histoUnfolded.Draw("PE SAME") # "E" ensures that error bars are drawn

    # legend
    legend = ROOT.TLegend(0.69, 0.74, 0.89, 0.89) # set coordinates as fractions of the canvas (x1, y1, x2, y2)
    legend.SetBorderSize(0) # removes the border around the legend box
    
    legend.AddEntry(histoUnfoldingTrue, "True Data", "l")
    legend.AddEntry(histoUnfoldingSmeared, "Smeared Data", "l")
    legend.AddEntry(histoPenultimate, f"{nIter - 1}. Iteration", "p")
    legend.AddEntry(histoUnfolded, f"{nIter}. Iteration", "p")
    
    legend.Draw()

    # save
    canvasUnfolding.SaveAs(f"img/{seedUnfolding}/{seedUnfolding}{abbrev}Unfolding.pdf") # save the canvas as a PDF file

    # ------------
    # CLOSURE TEST
    # ------------

    # define the histogram of the unfolded data and true data ratio
    histoRatio = histoUnfolded.Clone(f"closureRatio{abbrev}") # set the numerator = unfolded data of the ratio
    histoRatio.SetDirectory(0)
    histoRatio.Divide(histoUnfoldingTrue) # divide by the true data histogram

    # set up canvas
    canvasClosure = ROOT.TCanvas(f"canvasClosure{abbrev}", f"Closure Test for Unfolding of {name}", 800, 600)

    # customize the histogram
    histoRatio.SetTitle(f"Closure Test for Unfolding of {name};Values;Unfolded / True Data")

    histoRatio.SetMinimum(0.7)
    histoRatio.SetMaximum(1.3)

    histoRatio.SetLineColor(ROOT.kPink)
    histoRatio.SetMarkerStyle(ROOT.kFullCircle)
    histoRatio.SetMarkerColor(ROOT.kPink)
    histoRatio.SetMarkerSize(0.5)

    # draw the histogram
    histoRatio.Draw("PE") 
    
    # draw the reference line at y=1
    xMin = histoRatio.GetXaxis().GetXmin() # extract x-axis limits
    xMax = histoRatio.GetXaxis().GetXmax()
    line = ROOT.TLine(xMin, 1.0, xMax, 1.0) # create a line between xMin, xMax at y=1 (TLine(x1, y1, x2, y2))
    line.SetLineColor(ROOT.kGreen)
    line.SetLineStyle(2) # dashed line
    line.SetLineWidth(2)
    line.Draw("SAME")

    # save
    canvasClosure.SaveAs(f"img/{seedUnfolding}/{seedUnfolding}{abbrev}ClosureTest.pdf")

Using response matrix priors
Priors:

Vector (70)  is as follows

     |        1  |
------------------
   0 |0.000243503 
   1 |0.000854009 
   2 |0.00193052 
   3 |0.00347204 
   4 |0.00558056 
   5 |0.00829159 
   6 |0.0116136 
   7 |0.0153407 
   8 |0.0192262 
   9 |0.0236848 
  10 |0.0281288 
  11 |0.0322744 
  12 |0.0362824 
  13 |0.0396414 
  14 |0.0425695 
  15 |0.04494 
  16 |0.0465905 
  17 |0.047354 
  18 |0.047781 
  19 |0.0474005 
  20 |0.0461115 
  21 |0.0446655 
  22 |0.0427185 
  23 |0.0404399 
  24 |0.0377349 
  25 |0.0347969 
  26 |0.0320644 
  27 |0.0290093 
  28 |0.0262663 
  29 |0.0234523 
  30 |0.0206307 
  31 |0.0184077 
  32 |0.0156587 
  33 |0.0138162 
  34 |0.0119061 
  35 |0.0101521 
  36 |0.0086711 
  37 |0.00727508 
  38 |0.00609657 
  39 |0.00518456 
  40 |0.00421655 
  41 |0.00343654 
  42 |0.00285153 
  43 |0.00234353 
  44 |0.00184902 
  45 |0.00148552 
  46 |0.00121701 
  47 |0.00092851 
  48 |0.000736508 
  49 |0.000595507 
  50 |0.000461005 
  51 |0.

Info in <TCanvas::Print>: pdf file img/22/22NchUnfolding.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22NchClosureTest.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22S0pT1Unfolding.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22S0pT1ClosureTest.pdf has been created


### $p_\mathrm{T}$ Spectra

We will carry out two different methods of unfolding 2D $p_\mathrm{T}$ spectra:
1. **no unfolding at all (method A):** In this method, we assume that it is irrelevant whether we use true or smeared observable distribution as their shapes are almost the same. Such assumption is valid mainly for spherocity observables and invalid for multiplicity, as the shapes differ substantially. Additionally, we are sorting the events into relative (spherocity, multiplicity, ...) classes, e.g., top $1 \: \%$ $S_0^{p_\mathrm{T} = 1}$ events, which is equal to the most jetty events. The method would not work if we used absolute cuts, e.g., events with $S_0^{p_\mathrm{T} = 1} > 0.8$, because the true and smeared observable distributions still vary, and we would therefore lose events which satisfy the absolute cut on particle-level but not on detector-level. We only sort the particles into the classes, and then compute the value of the observable to obtain the absolute value of each cut.

2. **1D unfolding (method B):** For each $p_\mathrm{T}$ bin, we unfold the observable distribution (i.e., the $y$-axis in the 2D histogram).

We are going to need NumPy to perform operations to find quantiles later.

In [4]:
import numpy as np

Firstly, we load the 2D histograms generated in plot-ea.ipynb, unfolding matrices, and unfolded 1D distributions of observables from the previous section.

In [5]:
# load a 2D histograms of measured S0pT1 vs pT, true variant, and a S0pT1 unfolding matrix (UM)
histopTS0pT1True = fileUnfolding.Get(f"histo{seedUnfolding}pTS0pT1True")
histopTS0pT1Smeared = fileUnfolding.Get(f"histo{seedUnfolding}pTS0pT1Smeared")

UMS0pT1 = UMs["S0pT1"]
histoS0pT1Unfolded = histosUnfolded["S0pT1"]

We must ensure that the axis containing smeared data in the unfolding matrix is normalized in order that the values represent probability. Let's first check, whether the matrix is normalized in the $x$-axis.

In [6]:
nRows = UMS0pT1.GetNrows()
nCols = UMS0pT1.GetNcols()

# loop through each row
for jCol in range(nCols):
    
    # sum across all columns in a given row
    colSum = sum(UMS0pT1(iRow, jCol) for iRow in range(nRows))
    
    # print the sum
    print(f"Measured Bin {jCol} sum: {colSum:.4f}")

Measured Bin 0 sum: 0.0000
Measured Bin 1 sum: 8.5162
Measured Bin 2 sum: 8.0034
Measured Bin 3 sum: 12.6782
Measured Bin 4 sum: 11.6104
Measured Bin 5 sum: 8.5882
Measured Bin 6 sum: 7.7608
Measured Bin 7 sum: 7.5932
Measured Bin 8 sum: 7.2900
Measured Bin 9 sum: 6.7588
Measured Bin 10 sum: 6.1699
Measured Bin 11 sum: 6.2357
Measured Bin 12 sum: 5.8141
Measured Bin 13 sum: 5.1878
Measured Bin 14 sum: 5.1231
Measured Bin 15 sum: 5.0792
Measured Bin 16 sum: 4.9420
Measured Bin 17 sum: 4.7531
Measured Bin 18 sum: 4.5692
Measured Bin 19 sum: 4.3366
Measured Bin 20 sum: 4.2405
Measured Bin 21 sum: 4.1526
Measured Bin 22 sum: 4.0271
Measured Bin 23 sum: 4.0052
Measured Bin 24 sum: 3.8738
Measured Bin 25 sum: 3.6921
Measured Bin 26 sum: 3.6985
Measured Bin 27 sum: 3.6207
Measured Bin 28 sum: 3.5680
Measured Bin 29 sum: 3.4544
Measured Bin 30 sum: 3.4333
Measured Bin 31 sum: 3.3840
Measured Bin 32 sum: 3.3330
Measured Bin 33 sum: 3.2953
Measured Bin 34 sum: 3.1963
Measured Bin 35 sum: 3.1701


As we can see, the unfolding matrix is NOT normalized in the measured axis. Therefore, we must normalize it.

In [7]:
# load the function for x-axis normalization
ROOT.gInterpreter.ProcessLine('#include "cpp/normalizeUM.cpp"')

# normalize the UM in x-axis
ROOT.normalizeUM(UMS0pT1)

In [8]:
nRows = UMS0pT1.GetNrows()
nCols = UMS0pT1.GetNcols()

# loop through each row
for jCol in range(nCols):
    
    # sum across all columns in a given row
    colSum = sum(UMS0pT1(iRow, jCol) for iRow in range(nRows))
    
    # print the sum
    print(f"Measured Bin {jCol} sum: {colSum:.4f}")

Measured Bin 0 sum: 0.0000
Measured Bin 1 sum: 1.0000
Measured Bin 2 sum: 1.0000
Measured Bin 3 sum: 1.0000
Measured Bin 4 sum: 1.0000
Measured Bin 5 sum: 1.0000
Measured Bin 6 sum: 1.0000
Measured Bin 7 sum: 1.0000
Measured Bin 8 sum: 1.0000
Measured Bin 9 sum: 1.0000
Measured Bin 10 sum: 1.0000
Measured Bin 11 sum: 1.0000
Measured Bin 12 sum: 1.0000
Measured Bin 13 sum: 1.0000
Measured Bin 14 sum: 1.0000
Measured Bin 15 sum: 1.0000
Measured Bin 16 sum: 1.0000
Measured Bin 17 sum: 1.0000
Measured Bin 18 sum: 1.0000
Measured Bin 19 sum: 1.0000
Measured Bin 20 sum: 1.0000
Measured Bin 21 sum: 1.0000
Measured Bin 22 sum: 1.0000
Measured Bin 23 sum: 1.0000
Measured Bin 24 sum: 1.0000
Measured Bin 25 sum: 1.0000
Measured Bin 26 sum: 1.0000
Measured Bin 27 sum: 1.0000
Measured Bin 28 sum: 1.0000
Measured Bin 29 sum: 1.0000
Measured Bin 30 sum: 1.0000
Measured Bin 31 sum: 1.0000
Measured Bin 32 sum: 1.0000
Measured Bin 33 sum: 1.0000
Measured Bin 34 sum: 1.0000
Measured Bin 35 sum: 1.0000
Me

Now that the UM is properly normalized, we can proceed with the 1D unfolding. Since the observables are dependent on $p_\mathrm{T}$, we cannot just do a projection of the 2D histogram on the $y$-axis, as that would average out the $p_\mathrm{T}$ dependence. This is undesirable, because the observables are correlated with $p_\mathrm{T}$, e.g., highly spherical events will likely contain more high-$p_\mathrm{T}$ particles.

In [9]:
# load the C++ applyUM(histoMeas, UM) function which applies the UM onto the y-axis
ROOT.gInterpreter.ProcessLine('#include "cpp/applyUM.cpp"')

# calculate the 2D histogram of unfolded S0pT1 vs measured pT
histopTS0pT1UnfoldedB = ROOT.applyUM(histopTS0pT1Smeared, UMS0pT1)

# plot to visualize the 2D histogram of unfolded S0pT1 vs pT -- the labels and title is wrong, but its just for visualization
c1 = ROOT.TCanvas()
c1.SetLogz()
histopTS0pT1UnfoldedB.Draw("COLZ")
# UMS0pT1.Draw("colz")
c1.Draw()

Then, we obtain the $p_\mathrm{T}$ spectra for the top 1%, 5%, 10%, 90%, 95%, 99% $S_0^{p_\mathrm{T} = 1}$ events. 

In [10]:
# load the 1D event distributions of our observable -- we must not project the 2D histograms onto the y-axis to get the distribution of the observable, because that would give us particle distribution, not event
histoS0pT1True = fileUnfolding.Get(f"histo{seedUnfolding}S0pT1True")
histoS0pT1Smeared = fileUnfolding.Get(f"histo{seedUnfolding}S0pT1Smeared")

# define the quantile arrays with np.float64 data type to match ROOT's Double_t, which is required by the GetQuantiles() function
quantilesLow = np.array([0.01, 0.05, 0.1], dtype=np.float64)
quantilesHigh = np.array([0.9, 0.95, 0.99], dtype=np.float64)
quantilesDict = {False: quantilesLow, True: quantilesHigh} # low quantiles are assigned a boolean False, high quantiles True

# iterate through each quantile array
for kind, quantiles in quantilesDict.items():
    
    # define empty arrays for the output maximum S0pT1 values corresponding to the quantiles
    nQuantiles = len(quantiles)
    valuesTrue = np.zeros(nQuantiles, dtype=np.float64)
    valuesSmeared = np.zeros(nQuantiles, dtype=np.float64)
    valuesUnfolded = np.zeros(nQuantiles, dtype=np.float64)

    # calculate the S0pT1 values for the given quantiles, we also calculate them for the unfolded 1D distributions to get the absolute value of our relative cuts for the legend
    histoS0pT1True.GetQuantiles(nQuantiles, valuesTrue, quantiles)
    histoS0pT1Smeared.GetQuantiles(nQuantiles, valuesSmeared, quantiles)
    histoS0pT1Unfolded.GetQuantiles(nQuantiles, valuesUnfolded, quantiles)

    # get the pT spectra for each quantile
    for i in range(nQuantiles):
        quantile = quantiles[i]

        # extract bin numbers for each quantile
        valueTrue = valuesTrue[i]
        valueSmeared = valuesSmeared[i]
        valueUnfolded = valuesUnfolded[i] # this is the absolute value of the relative cut

        # find the bin number in the 1D distribtion
        binTrue1D = histoS0pT1True.GetXaxis().FindBin(valueTrue)
        binSmeared1D = histoS0pT1Smeared.GetXaxis().FindBin(valueSmeared)
        binUnfolded1D = histoS0pT1Unfolded.GetXaxis().FindBin(valueUnfolded)

        # find the bin number in the 2D histograms
        binTrue2D = histopTS0pT1True.GetYaxis().FindBin(valueTrue)
        binSmeared2D = histopTS0pT1Smeared.GetYaxis().FindBin(valueSmeared)
        binUnfolded2D = histopTS0pT1UnfoldedB.GetYaxis().FindBin(valueUnfolded)

        # calculate the x-axis projections 
        if kind:
            # this is executed for high quantiles
            histopTTrue = histopTS0pT1True.ProjectionX(f"histo{seedUnfolding}pTTrueTop{quantile}S0pT1", binTrue2D, -1)
            histopTTrue.SetDirectory(0)

            histopTSmeared = histopTS0pT1Smeared.ProjectionX(f"histo{seedUnfolding}pTSmearedTop{quantile}S0pT1", binSmeared2D, -1)
            histopTSmeared.SetDirectory(0)

            histopTUnfolded = histopTS0pT1UnfoldedB.ProjectionX(f"histo{seedUnfolding}pTUnfoldedTop{quantile}S0pT1", binUnfolded2D, -1)
            histopTUnfolded.SetDirectory(0)

        else:
            # this is executed for low quantiles
            histopTTrue = histopTS0pT1True.ProjectionX(f"histo{seedUnfolding}pTTrueTop{quantile}S0pT1", 1, binTrue2D)
            histopTTrue.SetDirectory(0)

            histopTSmeared = histopTS0pT1Smeared.ProjectionX(f"histo{seedUnfolding}pTSmearedTop{quantile}S0pT1", 1, binSmeared2D)
            histopTSmeared.SetDirectory(0)

            histopTUnfolded = histopTS0pT1UnfoldedB.ProjectionX(f"histo{seedUnfolding}pTUnfoldedTop{quantile}S0pT1", 1, binUnfolded2D)
            histopTUnfolded.SetDirectory(0)

        # ------------------------
        # PLOTTING -- NO UNFOLDING
        # ------------------------

        # set up canvas
        canvaspT = ROOT.TCanvas(f"canvasApTTop{quantile}S0pT1", "Method A: p_{T} spectrum for the top " + str(int(quantile * 100)) + "% S_{0}^{p_{T} = 1} events", 800, 600)
        canvaspT.SetLogy()

        # obtain per-event pT spectra by normalizing the pT histogram by the number of events
        histosS0pT1 = [histoS0pT1True, histoS0pT1Smeared, histoS0pT1Unfolded]
        histospT = [histopTTrue, histopTSmeared, histopTUnfolded]
        histosBins1D = [binTrue1D, binSmeared1D, binUnfolded1D]

        for histoS0pT1, histopT, bin1D in zip(histosS0pT1, histospT, histosBins1D):
            # we must distinguish between the low and high quantiles and calculate the number of events (the integral) in each quantile class
            if kind:
                integral = histoS0pT1.Integral(bin1D, -1)
            else:
                integral = histoS0pT1.Integral(1, bin1D)
            # normalize the pT spectrum
            if integral > 0:
                histopT.Scale(1.0 / integral)

        # customize the histograms
        histopTTrue.SetTitle("Method A: p_{T} spectrum for the top " + str(int(quantile * 100)) + "% S_{0}^{p_{T} = 1} events;p_{T};Event Normalized Counts")

        histopTTrue.SetLineColor(ROOT.kGreen)
        histopTSmeared.SetLineColor(ROOT.kBlue)

        # drawing
        histopTTrue.Draw("HIST")
        histopTSmeared.Draw("HIST SAME")

        # legend
        legend = ROOT.TLegend(0.69, 0.74, 0.89, 0.89)
        legend.SetBorderSize(0)
        
        legend.AddEntry(histopTTrue, "True Data", "l")
        legend.AddEntry(histopTSmeared, "Smeared Data", "l")

        legend.Draw()

        # text with S0pT1 cut value
        text = ROOT.TLatex()
        text.SetNDC() # tell ROOT to use Normalized Device Coordinates (NDC), so that we can set the text position by fractions of the plot size
        text.SetTextFont(42)
        text.SetTextSize(0.03)
        text.SetTextAlign(21)

        if kind:
            text.DrawLatex(0.5, 0.83, "unfolded S_{0}^{p_{T} = 1} > " + f"{valueUnfolded:.3f}")
        else:
            text.DrawLatex(0.5, 0.83, "unfolded S_{0}^{p_{T} = 1} < " + f"{valueUnfolded:.3f}")

        # save
        canvaspT.SaveAs(f"img/{seedUnfolding}/{seedUnfolding}ApTTop{quantile}S0pT1.pdf")

        # ----------------------------
        # CLOSURE TEST -- NO UNFOLDING
        # ----------------------------

        # define the histogram of the unfolded data and true data ratio
        histopTRatio = histopTSmeared.Clone(f"AclosureRatiopTTop{quantile}S0pT1") # set the numerator = unfolded data of the ratio
        histopTRatio.SetDirectory(0)
        histopTRatio.Divide(histopTTrue) # divide by the true data histogram

        # set up canvas
        canvaspTClosure = ROOT.TCanvas(f"canvasApTClosureTop{quantile}S0pT1", "Method A: Closure Test for Unfolding of a p_{T} spectrum for the top " + str(int(quantile * 100)) + "% S_{0}^{p_{T} = 1} events", 800, 600)

        # customize the histogram
        histopTRatio.SetTitle("Method A: Closure Test for Unfolding of p_{T} spectrum for the top " + str(int(quantile * 100)) + "% S_{0}^{p_{T} = 1} events;Values;Unfolded / True Data")

        # histopTRatio.SetMinimum(0.7)
        # histopTRatio.SetMaximum(1.3)

        histopTRatio.SetLineColor(ROOT.kPink)
        histopTRatio.SetMarkerStyle(ROOT.kFullCircle)
        histopTRatio.SetMarkerColor(ROOT.kPink)
        histopTRatio.SetMarkerSize(0.5)

        # draw the histogram
        histopTRatio.Draw("PE") 
        
        # draw the reference line at y=1
        xMin = histopTRatio.GetXaxis().GetXmin() # extract x-axis limits
        xMax = histopTRatio.GetXaxis().GetXmax()
        line = ROOT.TLine(xMin, 1.0, xMax, 1.0) # create a line between xMin, xMax at y=1 (TLine(x1, y1, x2, y2))
        line.SetLineColor(ROOT.kGreen)
        line.SetLineStyle(2) # dashed line
        line.SetLineWidth(2)
        line.Draw("SAME")

        # save
        canvaspTClosure.SaveAs(f"img/{seedUnfolding}/{seedUnfolding}ApTTop{quantile}S0pT1ClosureTest.pdf")

        # ------------------------
        # PLOTTING -- 1D UNFOLDING
        # ------------------------

        # set up canvas
        canvaspT = ROOT.TCanvas(f"canvasBpTTop{quantile}S0pT1", "Method B: p_{T} spectrum for the top " + str(int(quantile * 100)) + "% S_{0}^{p_{T} = 1} events", 800, 600)
        canvaspT.SetLogy()

        # customize the histograms
        histopTTrue.SetTitle("Method B: p_{T} spectrum for the top " + str(int(quantile * 100)) + "% S_{0}^{p_{T} = 1} events;p_{T};Event Normalized Counts")

        histopTTrue.SetLineColor(ROOT.kGreen)
        histopTSmeared.SetLineColor(ROOT.kBlue)
        histopTUnfolded.SetLineColor(ROOT.kRed)

        # drawing
        histopTTrue.Draw("HIST")
        histopTSmeared.Draw("HIST SAME")
        histopTUnfolded.Draw("HIST SAME")

        # legend
        legend = ROOT.TLegend(0.69, 0.74, 0.89, 0.89)
        legend.SetBorderSize(0)
        
        legend.AddEntry(histopTTrue, "True Data", "l")
        legend.AddEntry(histopTSmeared, "Smeared Data", "l")
        legend.AddEntry(histopTUnfolded, f"Unfolded Data", "l")
        
        legend.Draw()

        # text with S0pT1 cut value
        text = ROOT.TLatex()
        text.SetNDC() # tell ROOT to use Normalized Device Coordinates (NDC), so that we can set the text position by fractions of the plot size
        text.SetTextFont(42)
        text.SetTextSize(0.03)
        text.SetTextAlign(21)

        if kind:
            text.DrawLatex(0.5, 0.83, "unfolded S_{0}^{p_{T} = 1} > " + f"{valueUnfolded:.3f}")
        else:
            text.DrawLatex(0.5, 0.83, "unfolded S_{0}^{p_{T} = 1} < " + f"{valueUnfolded:.3f}")

        # save
        canvaspT.SaveAs(f"img/{seedUnfolding}/{seedUnfolding}BpTTop{quantile}S0pT1.pdf")

        # ----------------------------
        # CLOSURE TEST -- 1D UNFOLDING
        # ----------------------------

        # define the histogram of the unfolded data and true data ratio
        histopTRatio = histopTUnfolded.Clone(f"BclosureRatiopTTop{quantile}S0pT1") # set the numerator = unfolded data of the ratio
        histopTRatio.SetDirectory(0)
        histopTRatio.Divide(histopTTrue) # divide by the true data histogram

        # set up canvas
        canvaspTClosure = ROOT.TCanvas(f"canvasBpTClosureTop{quantile}S0pT1", "Method B: Closure Test for Unfolding of a p_{T} spectrum for the top " + str(int(quantile * 100)) + "% S_{0}^{p_{T} = 1} events", 800, 600)

        # customize the histogram
        histopTRatio.SetTitle("Method B: Closure Test for Unfolding of p_{T} spectrum for the top " + str(int(quantile * 100)) + "% S_{0}^{p_{T} = 1} events;Values;Unfolded / True Data")

        # histopTRatio.SetMinimum(0.7)
        # histopTRatio.SetMaximum(1.3)

        histopTRatio.SetLineColor(ROOT.kPink)
        histopTRatio.SetMarkerStyle(ROOT.kFullCircle)
        histopTRatio.SetMarkerColor(ROOT.kPink)
        histopTRatio.SetMarkerSize(0.5)

        # draw the histogram
        histopTRatio.Draw("PE") 
        
        # draw the reference line at y=1
        xMin = histopTRatio.GetXaxis().GetXmin() # extract x-axis limits
        xMax = histopTRatio.GetXaxis().GetXmax()
        line = ROOT.TLine(xMin, 1.0, xMax, 1.0) # create a line between xMin, xMax at y=1 (TLine(x1, y1, x2, y2))
        line.SetLineColor(ROOT.kGreen)
        line.SetLineStyle(2) # dashed line
        line.SetLineWidth(2)
        line.Draw("SAME")

        # save
        canvaspTClosure.SaveAs(f"img/{seedUnfolding}/{seedUnfolding}BpTTop{quantile}S0pT1ClosureTest.pdf")

Info in <TCanvas::Print>: pdf file img/22/22ApTTop0.01S0pT1.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22ApTTop0.01S0pT1ClosureTest.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22BpTTop0.01S0pT1.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22BpTTop0.01S0pT1ClosureTest.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22ApTTop0.05S0pT1.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22ApTTop0.05S0pT1ClosureTest.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22BpTTop0.05S0pT1.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22BpTTop0.05S0pT1ClosureTest.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22ApTTop0.1S0pT1.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22ApTTop0.1S0pT1ClosureTest.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22BpTTop0.1S0pT1.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22BpTTop0.1S0pT

In the end, we close the files we opened in the beginning.

In [11]:
fileTraining.Close()
fileUnfolding.Close()